# 3. LSTM next-day prediction

Predict **entire next day** using an **LSTM**: for each (hour, frequency, threshold, class), use the last **seq_len days** of au_pct as input and predict the next day's au_pct.

- **Error:** same as notebook 2 — computed over the entire day → **one MAE and one RMSE per day**; same testing data and error reporting structure.
- **Final visualization:** dropdown for **class**; show testing MAE/RMSE per day. **Final results table:** MAE (LSTM) per class.

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [15]:
# Path to final data (same as notebook 2)
work_dir = None
for candidate in [Path("organized"), Path("../organized"), Path("work_dir"), Path("../work_dir")]:
    fd = candidate / "final"
    if fd.exists():
        work_dir = candidate
        break
if work_dir is None:
    raise FileNotFoundError('Final dir not found. Tried: organized/final, ../organized/final, work_dir/final, ../work_dir/final')
final_dir = work_dir / "final"
training_dir = final_dir / "training"
testing_dir = final_dir / "testing"

In [16]:
def load_final_parquet(split_dir: Path, band: str, date_yyyymmdd: str) -> Optional[pd.DataFrame]:
    """Load final_<date>.parquet for a given split and band. Returns None if missing."""
    path = split_dir / band / f"final_{date_yyyymmdd}.parquet"
    if not path.exists():
        return None
    return pd.read_parquet(path)


def get_dates_and_bands(split_dir: Path):
    """Return sorted list of date_yyyymmdd and list of unique bands."""
    dates = set()
    bands = []
    for band_dir in sorted(split_dir.iterdir()):
        if not band_dir.is_dir():
            continue
        if band_dir.name not in bands:
            bands.append(band_dir.name)
        for p in band_dir.glob("final_*.parquet"):
            d = p.stem.replace("final_", "")
            dates.add(d)
    return sorted(dates), sorted(set(bands))

In [17]:
# Discover bands and dates (same as notebook 2)
training_dates, bands_training = get_dates_and_bands(training_dir)
testing_dates, bands_testing = get_dates_and_bands(testing_dir)
class_options = sorted(set(bands_training) | set(bands_testing))

### LSTM hyperparameters

In [18]:
# Sequence length: number of past days used to predict next day
SEQ_LEN = 14
# LSTM architecture
HIDDEN_SIZE = 32
NUM_LAYERS = 2
DROPOUT = 0.2
# Training
EPOCHS = 30
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
# Improve LSTM: normalize au_pct per band, add hour as feature (pure LSTM)
N_FEATURES = 2   # 1 = au_pct only; 2 = au_norm + hour/24 (recommended)
# With few days, use shorter seq_len to get more (X,y) samples per key (e.g. 6 days, seq=3 → 3 samples/key)
PREFER_MORE_SAMPLES = True   # when True and training_dates <= 7, cap effective_seq_len at 3
# Train one LSTM per (class, freq_center_ghz) for freq-specific dynamics; ~6*7=42 models, ~30–45 min
PER_FREQ_LSTM = True

### Optional: quick hyperparameter check (~2 min)
Run once to try a few configs on the first band only; then set SEQ_LEN / HIDDEN_SIZE above to the best and re-run full training.

In [19]:
# Optional: uncomment and run to compare a few configs on first band (short run)
# from tensorflow import keras
# _band = class_options[0]
# _eff = min(SEQ_LEN, len(training_dates)-1)
# if PREFER_MORE_SAMPLES and len(training_dates) <= 7: _eff = min(_eff, 3)
# _X, _y, _scaler, _keys = build_train_sequences(_band, _eff)
# if _X.size > 0:
#     for _seq, _hid in [(2, 16), (2, 32), (3, 16), (3, 32)]:
#         _s = min(_seq, len(training_dates)-1)
#         if _s < 1: continue
#         _X2, _y2, _, _ = build_train_sequences(_band, _s)
#         if _X2.size == 0: continue
#         _m = build_lstm_model(_s); _m.fit(_X2, _y2, epochs=10, batch_size=BATCH_SIZE, verbose=0)
#         _cache = {}; _pd, _mae, _ = compute_lstm_errors_one_band(_band, _m, _keys, _cache, _s, _scaler)
#         print(f"seq_len={_s} hidden={_hid} -> testing MAE: {_mae:.4f}")
# else:
#     print("No training data for first band.")


### Load day-series and build sequences

For each (hour, freq_center_ghz, threshold_dbm) we have a time series of au_pct by date. We use the last `SEQ_LEN` days to predict the next day's au_pct. One LSTM per band (shared across all keys).

In [20]:
def load_date_from_train_or_test(band: str, date_yyyymmdd: str) -> Optional[pd.DataFrame]:
    """Load a single day's data from training_dir or testing_dir."""
    df = load_final_parquet(training_dir, band, date_yyyymmdd)
    if df is None:
        df = load_final_parquet(testing_dir, band, date_yyyymmdd)
    return df


def build_day_series_for_band(band: str, dates: list[str]) -> tuple[pd.DataFrame, list[tuple]]:
    """
    Load all dates for band and build a panel: rows = (date, hour, freq_center_ghz, threshold_dbm), col = au_pct.
    Returns (panel with date index and multi-index columns for (hour, freq, thresh), key_cols list).
    Actually we return a dict: date -> df with columns hour, freq_center_ghz, threshold_dbm, au_pct.
    And we need the list of (hour, freq, threshold) keys in stable order.
    """
    key_cols = ["hour", "freq_center_ghz", "threshold_dbm"]
    by_date = {}
    keys_set = set()
    for d in dates:
        df = load_date_from_train_or_test(band, d)
        if df is None or df.empty:
            continue
        by_date[d] = df[key_cols + ["au_pct"]].copy()
        for _, row in df[key_cols].drop_duplicates().iterrows():
            keys_set.add((int(row["hour"]), float(row["freq_center_ghz"]), int(row["threshold_dbm"])))
    keys_list = sorted(keys_set)
    return by_date, keys_list


def build_train_sequences(band: str, seq_len: int, keys_subset: list = None) -> tuple[np.ndarray, np.ndarray, dict, list]:
    """Build X, y, scaler, keys_used. If keys_subset is set (e.g. one freq), train only on those keys."""
    by_date, keys_full = build_day_series_for_band(band, training_dates)
    keys_list = keys_subset if keys_subset is not None else keys_full
    empty_X = np.array([]).reshape(0, seq_len, N_FEATURES)
    empty_y = np.array([])
    scaler_fallback = {"min_au": 0.0, "max_au": 100.0}
    if len(by_date) < seq_len + 1:
        return empty_X, empty_y, scaler_fallback, keys_list
    ordered_dates = sorted(by_date.keys())
    X_list, y_list = [], []
    all_au = []
    for (hour, freq, thresh) in keys_list:
        series = []
        for d in ordered_dates:
            df = by_date[d]
            row = df[(df["hour"] == hour) & (df["freq_center_ghz"] == freq) & (df["threshold_dbm"] == thresh)]
            if row.empty:
                series.append(np.nan)
            else:
                series.append(float(row["au_pct"].iloc[0]))
        series = np.array(series, dtype=np.float64)
        if np.isnan(series).any():
            series = pd.Series(series).ffill().bfill().values
        all_au.extend(series.tolist())
        for i in range(seq_len, len(series)):
            x_slice = series[i - seq_len : i]
            y_val = series[i]
            X_list.append((x_slice, y_val, hour))
            y_list.append(y_val)
    if not X_list:
        return empty_X, empty_y, scaler_fallback, keys_list
    min_au = float(np.min(all_au))
    max_au = float(np.max(all_au))
    if max_au <= min_au:
        min_au, max_au = 0.0, 100.0
    scale = max_au - min_au + 1e-8
    scaler = {"min_au": min_au, "max_au": max_au}
    X_out, y_out = [], []
    for (x_slice, y_val, hour) in X_list:
        au_norm = (x_slice - min_au) / scale
        y_norm = (y_val - min_au) / scale
        if N_FEATURES == 2:
            feat = np.column_stack([au_norm, np.full(seq_len, hour / 24.0)]).astype(np.float32)
        else:
            feat = au_norm.reshape(-1, 1).astype(np.float32)
        X_out.append(feat)
        y_out.append(y_norm)
    return np.array(X_out), np.array(y_out, dtype=np.float32), scaler, keys_list

In [21]:
def build_lstm_model(seq_len: int, n_features: int = None):
    if n_features is None:
        n_features = N_FEATURES
    model = keras.Sequential()
    model.add(layers.Input(shape=(seq_len, n_features)))
    for i in range(NUM_LAYERS):
        model.add(layers.LSTM(HIDDEN_SIZE, return_sequences=(i < NUM_LAYERS - 1)))
        model.add(layers.Dropout(DROPOUT))
    model.add(layers.Dense(1))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss="mse", metrics=["mae"])
    return model

LSTM: stacked layers with dropout; output is next-day au_pct per sequence.

In [22]:
# build_lstm_model defined in cell above

### Predict next day for a given date

For each (hour, freq, threshold) we need the last SEQ_LEN days ending the day before the target. We use data from training and/or testing in chronological order.

In [23]:
def get_ordered_dates_up_to(target_date_yyyymmdd: str) -> list[str]:
    """All dates (training + testing) up to and including the day before target."""
    from datetime import datetime, timedelta
    try:
        dt = datetime.strptime(target_date_yyyymmdd, "%Y%m%d")
        prev = (dt - timedelta(days=1)).strftime("%Y%m%d")
    except Exception:
        return []
    all_dates = sorted(set(training_dates) | set(testing_dates))
    return [d for d in all_dates if d <= prev]


def predict_one_day(model: keras.Model, band: str, target_date: str, keys_list: list, by_date_cache: dict, seq_len: int, scaler: dict) -> Optional[pd.DataFrame]:
    """Predict au_pct for target_date; uses scaler for norm/inverse and N_FEATURES (au_norm + hour/24)."""
    ordered = get_ordered_dates_up_to(target_date)
    if len(ordered) < seq_len:
        return None
    use_dates = ordered[-seq_len:]
    key_cols = ["hour", "freq_center_ghz", "threshold_dbm"]
    min_au = scaler.get("min_au", 0.0)
    max_au = scaler.get("max_au", 100.0)
    scale = max_au - min_au + 1e-8
    sequences = []
    for (hour, freq, thresh) in keys_list:
        series = []
        for d in use_dates:
            if d not in by_date_cache:
                df = load_date_from_train_or_test(band, d)
                by_date_cache[d] = df[key_cols + ["au_pct"]].copy() if df is not None else None
            df = by_date_cache.get(d)
            if df is None or df.empty:
                series.append(np.nan)
                continue
            row = df[(df["hour"] == hour) & (df["freq_center_ghz"] == freq) & (df["threshold_dbm"] == thresh)]
            if row.empty:
                series.append(np.nan)
            else:
                series.append(float(row["au_pct"].iloc[0]))
        series = np.array(series, dtype=np.float64)
        if np.isnan(series).any():
            series = pd.Series(series).ffill().bfill().values
        sequences.append((series, hour))
    if not sequences:
        return None
    X_list = []
    for (series, hour) in sequences:
        au_norm = (series - min_au) / scale
        if N_FEATURES == 2:
            feat = np.column_stack([au_norm, np.full(seq_len, hour / 24.0)]).astype(np.float32)
        else:
            feat = au_norm.reshape(-1, 1).astype(np.float32)
        X_list.append(feat)
    X_batch = np.array(X_list)
    preds_norm = model.predict(X_batch, verbose=0)[:, 0]
    preds = preds_norm * scale + min_au
    rows = [{"hour": k[0], "freq_center_ghz": k[1], "threshold_dbm": k[2], "au_pct": float(preds[i])}
            for i, k in enumerate(keys_list)]
    return pd.DataFrame(rows)

In [ ]:
def compute_lstm_errors_one_band(band: str, model: keras.Model, keys_list: list, by_date_cache: dict, seq_len: int, scaler: dict) -> tuple[pd.DataFrame, float, float]:
    """For each testing date, predict and compute MAE/RMSE. scaler used for normalize/inverse in predict."""
    key_cols = ["hour", "freq_center_ghz", "threshold_dbm"]
    per_day_rows = []
    all_errs = []
    for date_curr in testing_dates:
        df_curr = load_final_parquet(testing_dir, band, date_curr)
        if df_curr is None or df_curr.empty:
            continue
        df_pred = predict_one_day(model, band, date_curr, keys_list, by_date_cache, seq_len, scaler)
        if df_pred is None or df_pred.empty:
            continue
        df_pred = df_pred.rename(columns={"au_pct": "au_pct_pred"})
        merge = df_curr.merge(df_pred, on=key_cols, how="inner")
        if merge.empty:
            continue
        err = merge["au_pct"] - merge["au_pct_pred"]
        mae = float(np.abs(err).mean())
        rmse = float(np.sqrt((err ** 2).mean()))
        per_day_rows.append({"date_yyyymmdd": date_curr, "MAE": mae, "RMSE": rmse})
        all_errs.extend(err.tolist())
    test_mae = float(np.abs(np.array(all_errs)).mean()) if all_errs else np.nan
    test_rmse = float(np.sqrt((np.array(all_errs) ** 2).mean())) if all_errs else np.nan
    per_day = pd.DataFrame(per_day_rows) if per_day_rows else pd.DataFrame(columns=["date_yyyymmdd", "MAE", "RMSE"])
    return per_day, test_mae, test_rmse


def compute_lstm_errors_one_band_per_freq(band: str, models_pf: dict, scalers_pf: dict, keys_pf: dict, seq_len: int) -> tuple[pd.DataFrame, float, float]:
    """When using one model per (band, freq): predict per freq, combine, then compute MAE/RMSE."""
    key_cols = ["hour", "freq_center_ghz", "threshold_dbm"]
    per_day_rows = []
    all_errs = []
    cache = {}
    for date_curr in testing_dates:
        df_curr = load_final_parquet(testing_dir, band, date_curr)
        if df_curr is None or df_curr.empty:
            continue
        pred_dfs = []
        for (b, freq) in models_pf:
            if b != band:
                continue
            model = models_pf[(b, freq)]
            scaler = scalers_pf.get((b, freq), {"min_au": 0, "max_au": 100})
            keys_list = keys_pf.get((b, freq), [])
            if not keys_list:
                continue
            df_p = predict_one_day(model, band, date_curr, keys_list, cache, seq_len, scaler)
            if df_p is not None and not df_p.empty:
                pred_dfs.append(df_p)
        if not pred_dfs:
            continue
        df_pred = pd.concat(pred_dfs, ignore_index=True).drop_duplicates(subset=key_cols)
        df_pred = df_pred.rename(columns={"au_pct": "au_pct_pred"})
        merge = df_curr.merge(df_pred, on=key_cols, how="inner")
        if merge.empty:
            continue
        err = merge["au_pct"] - merge["au_pct_pred"]
        per_day_rows.append({"date_yyyymmdd": date_curr, "MAE": float(np.abs(err).mean()), "RMSE": float(np.sqrt((err**2).mean()))})
        all_errs.extend(err.tolist())
    test_mae = float(np.abs(np.array(all_errs)).mean()) if all_errs else np.nan
    test_rmse = float(np.sqrt((np.array(all_errs)**2).mean())) if all_errs else np.nan
    per_day = pd.DataFrame(per_day_rows) if per_day_rows else pd.DataFrame(columns=["date_yyyymmdd", "MAE", "RMSE"])
    return per_day, test_mae, test_rmse

: 

### Train LSTM per band (or per band+freq) and evaluate on testing

In [ ]:
# Effective sequence length: with few days, use shorter seq to get more (X,y) samples per key
effective_seq_len = min(SEQ_LEN, len(training_dates) - 1)
if PREFER_MORE_SAMPLES and len(training_dates) <= 7:
    effective_seq_len = min(effective_seq_len, 3)
if effective_seq_len < 1:
    raise ValueError(f"Need at least 2 training dates; got {len(training_dates)}")
print(f"Training dates: {len(training_dates)}, effective_seq_len: {effective_seq_len}")
print(f"PER_FREQ_LSTM: {PER_FREQ_LSTM}")
import sys
sys.stdout.flush()

import time

class FlushEpochProgress(keras.callbacks.Callback):
    """Print and flush after each epoch so Jupyter shows progress live."""
    def __init__(self, total_epochs):
        self.total_epochs = total_epochs
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        loss = float(logs.get("loss", 0))
        mae = float(logs.get("mae", 0))
        print(f"  Epoch {epoch + 1}/{self.total_epochs} done — loss: {loss:.4f}, mae: {mae:.4f}")
        sys.stdout.flush()
        sys.stderr.flush()

n_bands = len(class_options)
results = {}
models = {}
keys_per_band = {}
model_seq_len = {}
scalers = {}
models_pf = {}
scalers_pf = {}
keys_pf = {}

if PER_FREQ_LSTM:
    for idx, band in enumerate(class_options):
        print(f"\n--- Band {idx + 1}/{n_bands}: {band} (per-freq LSTM) ---")
        sys.stdout.flush()
        _, keys_full = build_day_series_for_band(band, training_dates)
        freqs = sorted(set(k[1] for k in keys_full))
        t0 = time.perf_counter()
        for fi, freq in enumerate(freqs):
            keys_subset = [k for k in keys_full if k[1] == freq]
            X_train, y_train, scaler, keys_used = build_train_sequences(band, effective_seq_len, keys_subset)
            if X_train.size == 0:
                continue
            print(f"  Freq {fi + 1}/{len(freqs)}: {freq} GHz, {X_train.shape[0]} samples", end=" ... ")
            sys.stdout.flush()
            model = build_lstm_model(effective_seq_len)
            model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)
            models_pf[(band, freq)] = model
            scalers_pf[(band, freq)] = scaler
            keys_pf[(band, freq)] = keys_used
        print(f"  Evaluating on {len(testing_dates)} testing dates...")
        sys.stdout.flush()
        testing_per_day, test_mae, test_rmse = compute_lstm_errors_one_band_per_freq(band, models_pf, scalers_pf, keys_pf, effective_seq_len)
        elapsed = time.perf_counter() - t0
        print(f"  Done in {elapsed:.1f}s — testing MAE: {test_mae:.4f}, RMSE: {test_rmse:.4f}")
        sys.stdout.flush()
        results[band] = {"testing_per_day": testing_per_day, "testing_MAE": test_mae, "testing_RMSE": test_rmse}
else:
    for idx, band in enumerate(class_options):
        print(f"\n--- Band {idx + 1}/{n_bands}: {band} ---")
        sys.stdout.flush()
        sys.stderr.flush()
        X_train, y_train, scaler, keys_used = build_train_sequences(band, effective_seq_len)
        if X_train.size == 0:
            print(f"  Skipped (no training sequences)")
            sys.stdout.flush()
            results[band] = {"testing_per_day": pd.DataFrame(), "testing_MAE": np.nan, "testing_RMSE": np.nan}
            continue
        print(f"  Samples: {X_train.shape[0]}, training for {EPOCHS} epochs...")
        sys.stdout.flush()
        t0 = time.perf_counter()
        keys_per_band[band] = keys_used
        scalers[band] = scaler
        model = build_lstm_model(effective_seq_len)
        model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1, callbacks=[FlushEpochProgress(EPOCHS)])
        models[band] = model
        model_seq_len[band] = effective_seq_len
        cache = {}
        print(f"  Evaluating on {len(testing_dates)} testing dates...")
        sys.stdout.flush()
        testing_per_day, test_mae, test_rmse = compute_lstm_errors_one_band(band, model, keys_per_band[band], cache, effective_seq_len, scaler)
        elapsed = time.perf_counter() - t0
        print(f"  Done in {elapsed:.1f}s — testing MAE: {test_mae:.4f}, RMSE: {test_rmse:.4f}")
        sys.stdout.flush()
        results[band] = {"testing_per_day": testing_per_day, "testing_MAE": test_mae, "testing_RMSE": test_rmse}

final_results = pd.DataFrame([
    {"Class": band, "MAE": results[band]["testing_MAE"], "RMSE": results[band]["testing_RMSE"]}
    for band in class_options
])
print("LSTM next-day prediction — Final results (testing data)")
display(final_results.round(4))

Training dates: 6, effective_seq_len: 3
PER_FREQ_LSTM: True

--- Band 1/6: 195MHz (per-freq LSTM) ---
  Freq 1/84: 0.17425 GHz, 72 samples ...   Freq 2/84: 0.17475 GHz, 72 samples ...   Freq 3/84: 0.17525 GHz, 72 samples ...   Freq 4/84: 0.17575 GHz, 72 samples ...   Freq 5/84: 0.17625 GHz, 72 samples ...   Freq 6/84: 0.17675 GHz, 72 samples ...   Freq 7/84: 0.17725 GHz, 72 samples ...   Freq 8/84: 0.17775 GHz, 72 samples ...   Freq 9/84: 0.17825 GHz, 72 samples ...   Freq 10/84: 0.17875 GHz, 72 samples ...   Freq 11/84: 0.17925 GHz, 72 samples ...   Freq 12/84: 0.17975 GHz, 72 samples ...   Freq 13/84: 0.18025 GHz, 72 samples ...   Freq 14/84: 0.18075 GHz, 72 samples ...   Freq 15/84: 0.18125 GHz, 72 samples ...   Freq 16/84: 0.18175 GHz, 72 samples ...   Freq 17/84: 0.18225 GHz, 72 samples ...   Freq 18/84: 0.18275 GHz, 72 samples ...   Freq 19/84: 0.18325 GHz, 72 samples ...   Freq 20/84: 0.18375 GHz, 72 samples ...   Freq 21/84: 0.18425 GHz, 72 samples ...   Freq 22/84: 0.18475 GHz

### Final results table (same as naive: Class, MAE, RMSE)

In [ ]:
print("LSTM next-day prediction — Final results (testing data)")
display(final_results.round(4))

LSTM next-day prediction — Final results (testing data)


,Class,MAE,RMSE
0,195MHz,12.0677,28.6514
1,2441MHz,28.2511,38.7026
2,3765MHz,30.2048,40.6193
3,539MHz,12.8470,28.0156
4,5500MHz,3.1738,7.9297
5,915MHz,17.2830,24.6049


### Per-class view: dropdown for class — testing MAE/RMSE per day

In [ ]:
class_dropdown = widgets.Dropdown(
    options=class_options,
    value=class_options[0] if class_options else None,
    description="Class:",
    style={"description_width": "50px"},
)
out = widgets.Output()


def update_lstm_viz(class_band):
    with out:
        clear_output(wait=True)
        if class_band not in results:
            print(f"No results for class {class_band}")
            return
        r = results[class_band]
        test_df = r["testing_per_day"]
        test_mae = r["testing_MAE"]

        if not test_df.empty:
            test_display = test_df.copy()
            test_display["date"] = test_display["date_yyyymmdd"].str[:4] + "-" + test_display["date_yyyymmdd"].str[4:6] + "-" + test_display["date_yyyymmdd"].str[6:8]
            print(f"LSTM next-day prediction — Class: {class_band} (testing only)")
            display(test_display[["date", "MAE", "RMSE"]].round(4))
        print(f"Testing MAE (all testing days): {test_mae:.4f}")

        fig = go.Figure()
        if not test_df.empty:
            test_dates_dash = [f"{d[:4]}-{d[4:6]}-{d[6:8]}" for d in test_df["date_yyyymmdd"]]
            fig.add_trace(go.Bar(x=test_dates_dash, y=test_df["MAE"], name="MAE (per day)", marker_color="steelblue"))
        fig.update_layout(
            title=f"LSTM next-day MAE by date — {class_band}",
            xaxis_title="Date",
            yaxis_title="MAE",
            height=400,
        )
        fig.show()


widgets.interactive_output(update_lstm_viz, {"class_band": class_dropdown})
display(widgets.HBox([class_dropdown]), out)
update_lstm_viz(class_dropdown.value)

Output()

### Predicted vs Actual heatmaps

Pick a **class** and **testing date**. Left: actual AU (%) for that day. Right: LSTM prediction (same day). Same colorscale for comparison.